In [1]:
import os
import sys
import pickle
from pathlib import Path

from pyeCAP.io.tdt_io import cached_read_block

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import pyeCAP
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
import tdt

In [2]:
raw_ephys = pyeCAP.Ephys(r'D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804', stores = 'Wav2')
stim = pyeCAP.Stim(r'D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804')
cap = pyeCAP.ECAP(raw_ephys, stim)

#valid_tank = pyeCAP.Ephys(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352', stores = 'ECAP')

valid_ephys = pyeCAP.Ephys(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352', stores = 'ECAP')
valid_stim = pyeCAP.Stim(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352')
valid_cap = pyeCAP.ECAP(valid_ephys, valid_stim)

D:\PycharmProjects\pyeCAP\pyeCAP\io\tdt_io.py:162: UserWarning: No StoresListing file found, pyeCAP will assume default store names
  warnings.warn(
D:\PycharmProjects\pyeCAP\pyeCAP\io\tdt_io.py:233: UserWarning: StoresListing.txt could not be found. Defaulting Stimulation data to parameter store: eS1p, and raw stimulation store: eS1r.
  warnings.warn(
D:\PycharmProjects\pyeCAP\pyeCAP\ecap.py:48: UserWarning: Electrode distance information not provided. AUC calculation using standard neural windows cannot be completed.
  warnings.warn(
D:\PycharmProjects\pyeCAP\pyeCAP\ecap.py:48: UserWarning: Electrode distance information not provided. AUC calculation using standard neural windows cannot be completed.
  warnings.warn(


In [3]:
cap.dask_array(parameter = (0,0))

731689
16
900
813.0


ValueError: total size of new array must be unchanged

In [8]:
valid_cap.dask_array(parameter = (0,0))

8
365
3013.0


dask.array<transpose, shape=(365, 8, 3013), dtype=float32, chunksize=(261, 1, 3013), chunktype=numpy.ndarray>

In [5]:
cap.parameters.parameters

,,onset time (s),offset time (s),channel,frequency (Hz),stimulation gain,pulse count,duration (ms),period (ms),pulse amplitude A (μA),pulse duration A (ms),pulse amplitude B (μA),pulse duration B (ms),store
0,0,24.717066,54.687066,1,30.030031,1.0,900,29969.999313,33.299999,-3000.0,0.2,3000.0,0.2,eS1p


In [3]:
valid_cap.plot(channels = [0,1], parameters = (0,0))

16
900
813.0


ValueError: total size of new array must be unchanged

In [25]:
raw_ephys.

TypeError: 'Array' object is not callable

In [1]:
# Import Metadata and TDT Tanks
tankDF = pd.read_excel('D:\PycharmProjects\ENMapping_internal\ENMapping_Metadata.xlsx', sheet_name='PN1')
tankDF = tankDF.loc[ tankDF['Expt State'] == 'Mapping']

metaDF = pd.read_excel('D:\PycharmProjects\ENMapping_internal\ENMapping_Metadata.xlsx', sheet_name='Metadata')

#Create Dictionaries associating tanks with relevant IDS
L1_tanks = [tank for tank in tankDF.loc[tankDF['Longitudinal Position'] == 1, 'TDT Tank']]

filePATH = r'D:\Data\EN_Mapping\LM1_190109'
tdt_path_list = [filePATH + '\\' + tank for tank in L1_tanks]
#L1 = pyecap.Ephys(tdt_path_list, stores = 'Wav1')
L1_stim = pyeCAP.Stim(tdt_path_list)#, stores = 'eS1r')

NameError: name 'pd' is not defined

In [3]:
# Import good (/w StoresListing.txt) and bad (w/o StoresListing.txt)
#valid_tank = pyeCAP.Ephys(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352', stores = 'ECAP')
#invalid_tank = pyeCAP.Ephys(r'D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804', stores = 'Wav1')
invalid_stim = pyeCAP.Stim(r'D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804')

#good = tdt.read_block(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352')#, stores = 'ECAP')
#bad = tdt.read_block('D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804')

asdf


D:\PycharmProjects\pyeCAP\pyeCAP\io\tdt_io.py:162: UserWarning: No StoresListing file found, pyeCAP will assume default store names
  warnings.warn(


ValueError: No electrical stimulation detected

In [ ]:
# TdtIO StoresListing.txt metadata parsing
file = r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352\StoresListing.txt'
#txt = read(file)
with open(file, 'r') as f:
    txt = f.read()

metadata = {}
gizmo_dict = {}
obj_id = {}


# Parse StoresListing file
# This parsing is not ideal and should be improved to better account for all possible variations
for n, txtblock in enumerate(txt.split("\n\n")):
    # Read in experiment metadata text block
    if (
        n == 0 and "Experiment" in txtblock
    ):  # The first block of text should be the experiment information
        metadata.update({line.split(":")[0]: line.split(":")[1].strip() for line in txtblock.split("\n")} )
        metadata.pop("Time")
    # Read in storage data from each tdtgizmo
    elif txtblock.startswith("Object ID") or txtblock.startswith("ObjectID"):
        store_ids = []
        for txt_line in txtblock.split("\n"):
            if txt_line.startswith("Object ID") or txt_line.startswith(
                "ObjectID"
            ):
                object_id = txt_line.split("-")[0].split(":")[1].strip()
                gizmo_name = txt_line.split("-")[1].strip()
            elif txt_line.startswith(" Store ID") or txt_line.startswith(
                " StoreID"
            ):
                store_ids.append(txt_line.split(":")[1].strip())
        gizmo_dict.update({store_id: gizmo_name for store_id in store_ids})
        obj_id.update({store_id: object_id for store_id in store_ids})

#Pull metadata directly from tank when StoresListing.txt file does not exist
metadata["Gizmo Name"] = gizmo_dict  #Dictionary where key is the Store ID and Value is type of Gizmo used to collect
metadata["Gizmo ID"] = obj_id #Dictionary where key is the StoreID and value is the Object ID -- the "Name ID" of a data stream input into synapse

In [ ]:
"""
Metadata in TdtIO class assigns:
Experiment name (str) - Needed?
Subject name (str) - Needed?
User (str) - Needed?
Date(str) - Needed?
Gizmo Name (dict) - Needed? - Dictionary where key is the Store ID and Value is type of Gizmo used to collect - used by metadata in TdtStim
Gizmo ID (dict) - Dictionary where key is the StoreID and value is the Object ID -- the "Name ID" of a data stream input into synapse - Doesn't seem to be used by anything
"""
new_meta = {}
#Assign Date
bad.info.start_date.strftime('%m/%d/%Y')
new_meta.update({'Date' : bad.info.start_date.strftime('%m/%d/%Y')})

#Generate Gizmo Name Dictionary -- assume that eS1r is the stim driver stream


In [ ]:
bad.streams

In [ ]:
#Assign Date
x = bad.info.start_date

In [ ]:
x.strftime('%m/%d/%Y')

In [ ]:
gizmo_name = str.split("-")[1].strip()
gizmo_name

In [ ]:
good.stores

In [ ]:
from pyeCAP.io.tdt_io import cached_read_block
test = cached_read_block('D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804', headers = 1)

In [ ]:
test.stores()

In [5]:
from pyeCAP.io.tdt_io import TdtIO
test = TdtIO('D:\Data\EN_Mapping\LM1_190109\pnpig_010919-190109-122804')
#test = TdtIO(r'D:\Data\HF_Block\20240904_HFBlock_Pig05\HF_Block_Template-240904-083702\HF05-240904-113352')
test.stores

['Wav1',
 'IZn2',
 'Wav2',
 'eS1p',
 'eS1r',
 'Sgn_',
 'AmpA',
 'DurA',
 'AmpB',
 'DurB',
 'Tick',
 'Pper',
 'Pcnt']

In [7]:
test.metadata

D:\PycharmProjects\pyeCAP\pyeCAP\io\tdt_io.py:162: UserWarning: No StoresListing file found, pyeCAP will assume default store names
  warnings.warn(


{'Gizmo Name': {}, 'Gizmo ID': {}}

In [24]:
y = list(x.keys())

In [25]:
y.type

AttributeError: 'list' object has no attribute 'type'

In [32]:
|# Find pairs where all but the last character in a string match

#Generate new stores list with last character removed
truncated_stores = [store[:-1] for store in test.stores]

#Get duplicates from truncated stores list
dbl_stores = [store for store in set(truncated_stores) if truncated_stores.count(store) > 1]
dbl_stores

#Check doubles against original stores list to find ones end in -p and -s
for dbl in dbl_stores:

    store_pair = [store for store in test.stores if dbl in store]
    #print(store_pair)
    for store in store_pair:
        if store[-1] == 'p':
            print(store)
        elif store[-1] == 'r':
            print(store)
#all_stores = set(test.stores)

eS1p
eS1r


In [22]:
set(truncated_stores)

{'Amp', 'Dur', 'IZn', 'Pcn', 'Ppe', 'Sgn', 'Tic', 'Wav', 'eS1'}

In [26]:
store ='IZn'
truncated_stores.count(store)

1

In [17]:
d2  = {}
if len(d2) == 0:
    print('Dictionary is empty')

Dictionary is empty


In [ ]:
|